# 🧪 Data Science Project 1


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

print("✅ Libraries imported successfully")
print(f"Pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")


✅ Libraries imported successfully
Pandas  : 3.0.1
NumPy   : 2.0.1


In [2]:
np.random.seed(42)
N = 500

raw = pd.DataFrame({
    "employee_id":       range(1, N + 1),
    "age":               np.random.randint(22, 60, N).astype(float),
    "department":        np.random.choice(["Engineering","Sales","HR","Finance","Marketing"], N),
    "years_at_company":  np.random.randint(0, 20, N).astype(float),
    "monthly_salary":    np.random.normal(60000, 15000, N),
    "performance_score": np.random.choice([1,2,3,4,5], N).astype(float),
    "overtime_hours":    np.random.exponential(10, N),
    "distance_km":       np.random.exponential(15, N),
    "satisfaction":      np.random.uniform(1, 10, N),
    "left_company":      np.random.choice([0, 1], N, p=[0.7, 0.3]),  # TARGET
})

# Inject missingness
raw.loc[np.random.choice(N, int(N*0.03), replace=False), "age"]               = np.nan  # 3%
raw.loc[np.random.choice(N, int(N*0.10), replace=False), "monthly_salary"]    = np.nan  # 10%
raw.loc[np.random.choice(N, int(N*0.08), replace=False), "performance_score"] = np.nan  # 8%
raw.loc[np.random.choice(N, int(N*0.25), replace=False), "distance_km"]       = np.nan  # 25%

# Inject outliers
outlier_idx = np.random.choice(N, 15, replace=False)
raw.loc[outlier_idx[:8],  "monthly_salary"] = np.random.uniform(300000, 500000, 8)
raw.loc[outlier_idx[8:],  "overtime_hours"] = np.random.uniform(200, 400, 7)

print(f"Dataset shape : {raw.shape}")
raw.head()


Dataset shape : (500, 10)


,employee_id,age,department,years_at_company,monthly_salary,performance_score,overtime_hours,distance_km,satisfaction,left_company
0,1,50.0,Sales,5.0,84397.792075,5.0,12.127908,4.153967,6.046105,0
1,2,36.0,Sales,0.0,22780.575202,2.0,2.710655,10.415326,6.655838,0
2,3,29.0,Marketing,16.0,67707.527896,2.0,7.181856,17.073728,7.197685,0
3,4,42.0,Marketing,10.0,56147.845374,5.0,1.100658,1.189532,3.279451,0
4,5,40.0,Marketing,17.0,NaN,5.0,4.853387,4.817628,1.089809,1


In [3]:
print("=== Basic Info ===")
raw.info()


=== Basic Info ===
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   employee_id        500 non-null    int64  
 1   age                485 non-null    float64
 2   department         500 non-null    str    
 3   years_at_company   500 non-null    float64
 4   monthly_salary     450 non-null    float64
 5   performance_score  460 non-null    float64
 6   overtime_hours     500 non-null    float64
 7   distance_km        375 non-null    float64
 8   satisfaction       500 non-null    float64
 9   left_company       500 non-null    int64  
dtypes: float64(7), int64(2), str(1)
memory usage: 42.6 KB


In [4]:
print("=== Descriptive Statistics ===")
raw.describe().round(2)


=== Descriptive Statistics ===


,employee_id,age,years_at_company,monthly_salary,performance_score,overtime_hours,distance_km,satisfaction,left_company
count,500.00,485.0,500.00,450.00,460.00,500.00,375.00,500.00,500.00
mean,250.50,41.4,9.51,66564.69,3.05,13.69,14.90,5.51,0.26
std,144.48,11.1,5.77,48046.49,1.40,37.41,14.54,2.59,0.44
min,1.00,22.0,0.00,20852.37,1.00,0.00,0.02,1.01,0.00
25%,125.75,32.0,4.00,49534.68,2.00,3.11,4.06,3.21,0.00
50%,250.50,43.0,10.00,61164.98,3.00,6.53,9.90,5.51,0.00
75%,375.25,51.0,15.00,72330.61,4.00,13.41,20.95,7.83,1.00
max,500.00,59.0,19.00,493641.77,5.00,396.57,74.98,10.00,1.00


In [5]:
print("=== Missing Values ===")
missing = raw.isnull().sum()
missing_pct = (raw.isnull().mean() * 100).round(2)
pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})[missing > 0]


=== Missing Values ===


,Missing Count,Missing %
age,15,3.0
monthly_salary,50,10.0
performance_score,40,8.0
distance_km,125,25.0


In [6]:
print("=== Target Distribution ===")
raw["left_company"].value_counts().rename({0:"Stayed", 1:"Left"})


=== Target Distribution ===


left_company
Stayed    370
Left      130
Name: count, dtype: int64

In [7]:
from sklearn.impute import KNNImputer

df = raw.copy()

for col in df.columns:
    miss_pct = df[col].isnull().mean()
    if miss_pct == 0 or col in ("employee_id", "department", "left_company"):
        continue

    if miss_pct < 0.05:
        before = len(df)
        df.dropna(subset=[col], inplace=True)
        df.reset_index(drop=True, inplace=True)
        print(f"  {col:22s} | {miss_pct:.0%} → DROP ROWS       | removed {before - len(df)} rows")

    elif miss_pct <= 0.20:
        if df[col].skew() > 1.0:
            fill_val = df[col].median()
            df[col].fillna(fill_val, inplace=True)
            print(f"  {col:22s} | {miss_pct:.0%} → GLOBAL MEDIAN   | filled with {fill_val:.2f}")
        else:
            df[col] = df.groupby("department")[col].transform(lambda x: x.fillna(x.median()))
            print(f"  {col:22s} | {miss_pct:.0%} → GROUP MEDIAN    | by department")

    else:
        num_cols = df.select_dtypes(include=np.number).columns.tolist()
        imputer  = KNNImputer(n_neighbors=5)
        df[num_cols] = pd.DataFrame(imputer.fit_transform(df[num_cols]),
                                    columns=num_cols, index=df.index)
        print(f"  {col:22s} | {miss_pct:.0%} → KNN IMPUTATION  | k=5")

print(f"\n✅ Missing values remaining: {df.isnull().sum().sum()}")
print(f"✅ Shape after imputation   : {df.shape}")


  age                    | 3% → DROP ROWS       | removed 15 rows
  monthly_salary         | 10% → GLOBAL MEDIAN   | filled with 61164.98
  performance_score      | 8% → GROUP MEDIAN    | by department
  distance_km            | 25% → KNN IMPUTATION  | k=5

✅ Missing values remaining: 0
✅ Shape after imputation   : (485, 10)


In [8]:
outlier_cols = ["monthly_salary", "overtime_hours", "distance_km"]

for col in outlier_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = np.clip(df[col], lower, upper)
    print(f"  {col:22s} | bounds [{lower:.1f}, {upper:.1f}] | {n_out} outliers winsorized")

print(f"\n✅ Row count preserved: {len(df)}")


  monthly_salary         | bounds [20229.8, 101903.9] | 13 outliers winsorized
  overtime_hours         | bounds [-11.9, 28.0] | 40 outliers winsorized
  distance_km            | bounds [-17.1, 41.4] | 30 outliers winsorized

✅ Row count preserved: 485


In [9]:
# Feature 1: Salary per Year of Experience
df["salary_per_year"] = df["monthly_salary"] / (df["years_at_company"] + 1)
print("✅ Feature 1: salary_per_year = monthly_salary / (years_at_company + 1)")

# Feature 2: Work-Life Balance Score
df["work_life_score"] = (df["satisfaction"] * 10) / (df["overtime_hours"] + 1)
print("✅ Feature 2: work_life_score = (satisfaction × 10) / (overtime_hours + 1)")

# Feature 3: Commute Burden Index (0–1 scaled)
df["commute_burden"] = (df["distance_km"] / df["distance_km"].max()).round(4)
print("✅ Feature 3: commute_burden  = distance_km / max(distance_km)")

# Feature 4: Performance Efficiency
df["performance_efficiency"] = df["performance_score"] / (df["years_at_company"] + 1)
print("✅ Feature 4: perf_efficiency = performance_score / (years_at_company + 1)")

# Feature 5: Late Hire Flag
df["late_hire_flag"] = ((df["age"] > 40) & (df["years_at_company"] < 3)).astype(int)
print("✅ Feature 5: late_hire_flag  = (age > 40) & (years_at_company < 3)")

print("\nNew feature preview:")
df[["salary_per_year","work_life_score","commute_burden",
    "performance_efficiency","late_hire_flag"]].describe().round(3)


✅ Feature 1: salary_per_year = monthly_salary / (years_at_company + 1)
✅ Feature 2: work_life_score = (satisfaction × 10) / (overtime_hours + 1)
✅ Feature 3: commute_burden  = distance_km / max(distance_km)
✅ Feature 4: perf_efficiency = performance_score / (years_at_company + 1)
✅ Feature 5: late_hire_flag  = (age > 40) & (years_at_company < 3)

New feature preview:


,salary_per_year,work_life_score,commute_burden,performance_efficiency,late_hire_flag
count,485.000,485.000,485.000,485.000,485.000
mean,10760.418,12.091,0.330,0.534,0.089
std,13140.916,14.809,0.276,0.692,0.285
min,1226.610,0.422,0.000,0.050,0.000
25%,3977.552,2.993,0.117,0.176,0.000
50%,6123.595,6.784,0.230,0.286,0.000
75%,10754.306,14.443,0.470,0.556,0.000
max,96200.909,97.785,1.000,5.000,1.000


In [10]:
df = pd.get_dummies(df, columns=["department"], drop_first=True, dtype=int)
ohe_cols = [c for c in df.columns if "department" in c]
print(f"✅ One-Hot columns created: {ohe_cols}")
df[ohe_cols].head()


✅ One-Hot columns created: ['department_Finance', 'department_HR', 'department_Marketing', 'department_Sales']


,department_Finance,department_HR,department_Marketing,department_Sales
0,0,0,0,1
1,0,0,0,1
2,0,0,1,0
3,0,0,1,0
4,0,0,1,0


In [11]:
TARGET = "left_company"
feature_cols = [c for c in df.select_dtypes(include=np.number).columns
                if c not in (TARGET, "employee_id")]

corr_matrix = df[feature_cols].corr().abs()
upper_tri   = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

print("Correlation matrix (absolute values):")
print(corr_matrix.round(3).to_string())


Correlation matrix (absolute values):
                          age  years_at_company  monthly_salary  performance_score  overtime_hours  distance_km  satisfaction  salary_per_year  work_life_score  commute_burden  performance_efficiency  late_hire_flag  department_Finance  department_HR  department_Marketing  department_Sales
age                     1.000             0.048           0.003              0.043           0.018        0.098         0.030            0.035            0.046           0.098                   0.010           0.280               0.016          0.010                 0.016             0.044
years_at_company        0.048             1.000           0.073              0.031           0.045        0.014         0.022            0.679            0.015           0.014                   0.640           0.450               0.010          0.007                 0.079             0.006
monthly_salary          0.003             0.073           1.000              0.024       

In [12]:
threshold = 0.80
to_drop   = []

for col in upper_tri.columns:
    if any(upper_tri[col] > threshold):
        partners = upper_tri.index[upper_tri[col] > threshold].tolist()
        for partner in partners:
            corr_col     = abs(df[col].corr(df[TARGET]))
            corr_partner = abs(df[partner].corr(df[TARGET]))
            weaker = col if corr_col < corr_partner else partner
            if weaker not in to_drop:
                to_drop.append(weaker)
                print(f"  Pair ({col} ↔ {partner})  r={upper_tri.loc[partner,col]:.3f}"
                      f"  → dropping '{weaker}' (weaker target correlation)")

if not to_drop:
    print("✅ No collinear pairs above threshold.")
else:
    df.drop(columns=to_drop, inplace=True)
    print(f"\n✅ Dropped {len(to_drop)} column(s): {to_drop}")

print(f"\nFinal features ({len(df.columns)-2}):")
print([c for c in df.columns if c not in ("employee_id", TARGET)])


  Pair (commute_burden ↔ distance_km)  r=1.000  → dropping 'distance_km' (weaker target correlation)
  Pair (performance_efficiency ↔ salary_per_year)  r=0.809  → dropping 'salary_per_year' (weaker target correlation)

✅ Dropped 2 column(s): ['distance_km', 'salary_per_year']

Final features (14):
['age', 'years_at_company', 'monthly_salary', 'performance_score', 'overtime_hours', 'satisfaction', 'work_life_score', 'commute_burden', 'performance_efficiency', 'late_hire_flag', 'department_Finance', 'department_HR', 'department_Marketing', 'department_Sales']


In [14]:
import pandera as pa
from pandera import Column, DataFrameSchema, Check

schema_cols = {
    "employee_id":          Column(int,   Check.ge(1),          nullable=False),
    "age":                  Column(float, Check.between(18,75),  nullable=False),
    "years_at_company":     Column(float, Check.ge(0),           nullable=False),
    "monthly_salary":       Column(float, Check.ge(0),           nullable=False),
    "performance_score":    Column(float, Check.between(1,5),    nullable=False),
    "overtime_hours":       Column(float, Check.ge(0),           nullable=False),
    "satisfaction":         Column(float, Check.between(1,10),   nullable=False),
    "work_life_score":      Column(float, Check.ge(0),           nullable=False),
    "commute_burden":       Column(float, Check.between(0,1),    nullable=False),
    "left_company":         Column(int,   Check.isin([0,1]),     nullable=False),
}

# Only validate columns present after collinearity eradication
schema_cols = {k: v for k, v in schema_cols.items() if k in df.columns}
schema = DataFrameSchema(columns=schema_cols, coerce=True)

df["employee_id"]  = df["employee_id"].astype(int)
df["left_company"] = df["left_company"].astype(int)

try:
    validated_df = schema.validate(df, lazy=True)
    print("✅ Pandera schema validation PASSED — all contracts satisfied!")
except pa.errors.SchemaErrors as e:
    print(f"⚠ Schema failures: {len(e.failure_cases)} cases")
    print(e.failure_cases)
    validated_df = df


✅ Pandera schema validation PASSED — all contracts satisfied!


In [15]:
import os
os.makedirs("feature_store", exist_ok=True)

# Offline Store (Parquet)
validated_df.to_parquet("feature_store/offline_features.parquet", index=False)
print(f"✅ Offline store → feature_store/offline_features.parquet")

# Online Store (CSV snapshot)
snap_cols = ["employee_id","work_life_score","commute_burden",
             "performance_efficiency","late_hire_flag","left_company"]
snap_cols = [c for c in snap_cols if c in validated_df.columns]
validated_df[snap_cols].to_csv("feature_store/online_features.csv", index=False)
print(f"✅ Online store  → feature_store/online_features.csv")


✅ Offline store → feature_store/offline_features.parquet
✅ Online store  → feature_store/online_features.csv


## 📊 Final Summary Report

In [16]:
print("=" * 55)
print("  PIPELINE SUMMARY REPORT")
print("=" * 55)
print(f"  Raw rows            : {N}")
print(f"  Rows after cleaning : {len(validated_df)}")
print(f"  Missing values left : {validated_df.isnull().sum().sum()}")
print(f"  Engineered features : 5")
print(f"  Final feature cols  : {len(validated_df.columns) - 2}")
print(f"  Pandera schema      : PASSED")
print(f"  Offline store       : WRITTEN (Parquet)")
print(f"  Online store        : WRITTEN (CSV)")
print("=" * 55)
print("\nTarget Distribution:")
vc = validated_df["left_company"].value_counts()
print(f"  Stayed (0) : {vc.get(0,0)} employees")
print(f"  Left   (1) : {vc.get(1,0)} employees")
print("\n✅ Project 1 Complete — Dataset is ML-Ready!")


  PIPELINE SUMMARY REPORT
  Raw rows            : 500
  Rows after cleaning : 485
  Missing values left : 0
  Engineered features : 5
  Final feature cols  : 14
  Pandera schema      : PASSED
  Offline store       : WRITTEN (Parquet)
  Online store        : WRITTEN (CSV)

Target Distribution:
  Stayed (0) : 358 employees
  Left   (1) : 127 employees

✅ Project 1 Complete — Dataset is ML-Ready!


In [17]:
# Final preview
validated_df.head(10)


,employee_id,age,years_at_company,monthly_salary,performance_score,overtime_hours,satisfaction,left_company,work_life_score,commute_burden,performance_efficiency,late_hire_flag,department_Finance,department_HR,department_Marketing,department_Sales
0,1,50.0,5.0,84397.792075,5.0,12.127908,6.046105,0,4.605535,0.1003,0.833333,0,0,0,0,1
1,2,36.0,0.0,22780.575202,2.0,2.710655,6.655838,0,17.937097,0.2515,2.000000,0,0,0,0,1
2,3,29.0,16.0,67707.527896,2.0,7.181856,7.197685,0,8.797130,0.4122,0.117647,0,0,0,1,0
3,4,42.0,10.0,56147.845374,5.0,1.100658,3.279451,0,15.611544,0.0287,0.454545,0,0,0,1,0
4,5,40.0,17.0,60132.609243,5.0,4.853387,1.089809,1,1.861843,0.1163,0.277778,0,0,0,1,0
5,6,44.0,16.0,82515.582600,3.0,6.687806,7.510993,0,9.770008,0.5957,0.176471,0,0,1,0,0
6,7,32.0,6.0,65482.601022,3.0,10.561974,5.820909,1,5.034529,0.2229,0.428571,0,0,0,1,0
7,8,32.0,16.0,69243.505872,2.0,28.026750,8.525059,0,2.936967,0.5232,0.117647,0,0,0,0,0
8,9,45.0,19.0,51391.657765,3.0,9.179197,8.392495,1,8.244752,0.1145,0.150000,0,1,0,0,0
9,10,57.0,8.0,53034.133851,4.0,13.609928,8.589936,1,5.879519,0.0163,0.444444,0,0,0,0,0
